# Multi-Speaker Dialogue Generator

Run the setup cell below to install required system packages and Python dependencies before running the application.

In [ ]:
# Setup cell for Google Colab environment
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q edge-tts pydub ipywidgets nest_asyncio

In [ ]:
import asyncio
import os
import nest_asyncio
import edge_tts
import ipywidgets as widgets
from pydub import AudioSegment
from IPython.display import Audio, display, clear_output

# Apply nest_asyncio to support nested event loops in Jupyter environments
nest_asyncio.apply()

# ==============================================================================
# VOICE CONFIGURATION
# ==============================================================================
MALE_VOICE = "en-GB-RyanNeural"     # British Male (Speaker 1)
FEMALE_VOICE = "en-GB-SoniaNeural"  # British Female (Speaker 2)
SPEECH_RATE = "+10%"                # Speed (+0%, +10%, +15%)

# Create UI Widgets
text_input = widgets.Textarea(
    value='',
    placeholder='Paste your "Speaker: Dialogue" script here...\nExample:\nJohn: Hi Mary!\nMary: Hello John!',
    description='Script:',
    layout=widgets.Layout(width='98%', height='200px')
)

button = widgets.Button(
    description='Generate Audio',
    button_style='primary',
    icon='play'
)

output_area = widgets.Output()

display(text_input, button, output_area)

async def generate_speech_segment(text, voice, rate, output_filename):
    communicate = edge_tts.Communicate(text=text, voice=voice, rate=rate)
    await communicate.save(output_filename)

def generate_audio_event(b):
    with output_area:
        clear_output()
        script_text = text_input.value.strip()
        
        if not script_text:
            print("Please paste a script into the text box above first.")
            return

        # Parse script lines into speaker and dialogue
        lines = [line.strip() for line in script_text.split('\n') if line.strip()]
        parsed_dialogue = []

        for line in lines:
            if ":" in line:
                speaker, text = line.split(":", 1)
                parsed_dialogue.append((speaker.strip(), text.strip()))

        if not parsed_dialogue:
            print("No valid lines found. Ensure each line is formatted as 'SpeakerName: Dialogue'")
            return

        # Map distinct speakers to Male and Female neural voices
        unique_speakers = list(dict.fromkeys([s for s, _ in parsed_dialogue]))
        voice_map = {}

        if len(unique_speakers) >= 1:
            voice_map[unique_speakers[0]] = MALE_VOICE
        if len(unique_speakers) >= 2:
            voice_map[unique_speakers[1]] = FEMALE_VOICE

        print(f"Detected Speakers: {unique_speakers}")
        print("Generating multi-speaker British Neural audio...")

        combined_audio = AudioSegment.empty()
        pause = AudioSegment.silent(duration=350)  # 0.35s gap between turns

        for idx, (speaker, text) in enumerate(parsed_dialogue):
            filename = f"temp_edge_{idx}.mp3"
            voice = voice_map.get(speaker, MALE_VOICE)
            
            try:
                # Run native Python async API safely in Colab/Jupyter
                loop = asyncio.get_event_loop()
                loop.run_until_complete(generate_speech_segment(text, voice, SPEECH_RATE, filename))
                
                if os.path.exists(filename):
                    segment = AudioSegment.from_mp3(filename)
                    combined_audio += segment + pause
                    os.remove(filename)
            except Exception as e:
                print(f"Error generating segment for {speaker} (line {idx+1}): {e}")

        output_file = "ielts_listening_dialogue.mp3"
        combined_audio.export(output_file, format="mp3")

        print("\nAudio ready! Click play below:\n")
        display(Audio(output_file, autoplay=False))

button.on_click(generate_audio_event)